In [ ]:
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as f
from  torchvision import transforms as t,datasets
from torch.utils.data import Dataset, DataLoader
import time
from tqdm import tqdm
d=torch.device("cpu")
import pandas as pd
import numpy as np

In [5]:

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.11.0+cpu
None
False


AssertionError: Torch not compiled with CUDA enabled

In [2]:
#data set;
data=torch.load("mathbert_embeddings.pt")
print(data.shape)

torch.Size([6525, 768])


In [66]:
import math
embeddings=data
pt=pd.read_csv("dataset_all_four.csv")
y=pt[["label"]].values
# y_min= y.min(axis=0)
# y_max= y.max(axis=0)
# y_norm = (y-y_min)/(y_max - y_min)
y=torch.tensor(y,dtype=torch.float32)
print(y.shape)
class EmbeddingDataset(Dataset):
    def __init__(self, embeddings, labels):
        self.embeddings = embeddings
        self.labels = labels
    def __len__(self):
        return len(self.embeddings)
    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]
dataset = EmbeddingDataset(embeddings, y)
train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, val_size, test_size])
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}, Test size: {len(test_dataset)}")


torch.Size([6525, 1])
Train size: 4567, Val size: 978, Test size: 980


In [51]:
print(y)

tensor([[0.],
        [0.],
        [1.],
        ...,
        [0.],
        [1.],
        [1.]])


In [52]:
#hyperparameters
embeddings_size=768
num_heads=8
learning_rate=0.001
batch_size=64

In [53]:
# class self_att(nn.Module):
#     def __init__(self,embed_size=128,heads=12):
#         super(self_att,self).__init__()
#         self.embed_size=embed_size
#         self.head=heads
#         self.head_dim=embed_size//self.head
#         self.qvk=nn.Linear(embed_size,embed_size*3,bias=False)
#         self.fc_out=nn.Linear(embed_size,embed_size)
#     def forward(self,x):
#         B,L,_=x.shape
#         out=self.qvk(x)
#         chunks = torch.chunk(out, chunks=3, dim=-1)
#         q=chunks[0]
#         k=chunks[1]
#         v=chunks[2]
#         q = q.view(B, L, self.head, self.head_dim).transpose(1, 2)
#         k = k.view(B, L, self.head, self.head_dim).transpose(1, 2)
#         v = v.view(B, L, self.head, self.head_dim).transpose(1, 2)
#         scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
#         attn = torch.softmax(scores, dim=-1)
#         out = attn @ v
#         out = out.transpose(1, 2).contiguous().view(B, L, self.embed_size)
#         out=self.fc_out(out)
#         return out
class EquationEncoder(nn.Module):

    def __init__(self, embed_size=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_size, 256),
            nn.ReLU(),
            nn.LayerNorm(256),
            nn.Linear(256, embed_size)
        )
    def forward(self, x):
        return self.net(x)
class encoder_block(nn.Module):
    def __init__(self,embedding,head):
        super().__init__()
        self.s_a=EquationEncoder(embedding)
        self.layer_norm=nn.LayerNorm(embedding)
        self.layer_norm2=nn.LayerNorm(embedding)
        self.layer_1=nn.Linear(embedding,embedding*2)
        self.layer_2=nn.Linear(embedding*2,embedding)
        self.dropout=nn.Dropout(0.3)
    def forward(self,x):
        out=self.s_a(x)
        out=out+x
        out=self.layer_norm(out)
        out_nn=self.layer_1(out)
        out_nn=self.dropout(out_nn)
        out_nn=nn.functional.relu(out_nn)
        out_nn=self.layer_2(out_nn)
        out_nn=self.dropout(out_nn)
        out=out+out_nn
        out=self.layer_norm2(out)
        return out
class encoder_layer(nn.Module):
    def __init__(self,embedding,head,layers):
        super().__init__()
        self.layers=nn.ModuleList([encoder_block(embedding,head) for _ in range(layers)])
    def forward(self,x):
        for layer in self.layers:
            x=layer(x)
        return x
    
class p_cf_simple(nn.Module):
    def __init__(self,embedding):
        super().__init__()
        self.wp_wcf=nn.Linear(embedding,4*embedding)
        self.layer_norm=nn.LayerNorm(embedding)
        self.layer_1_p=nn.Linear(embedding*2,embedding*4)
        self.layer_2_p=nn.Linear(embedding*4,embedding*2)
        self.layer_3_p=nn.Linear(embedding*2,embedding)
        self.layer_4_p=nn.Linear(embedding,1)
        # self.layer_1_c=nn.Linear(embedding*2,embedding*4)
        # self.layer_2_c=nn.Linear(embedding*4,embedding*2)
        # self.layer_3_c=nn.Linear(embedding*2,embedding)
        # self.layer_4_c=nn.Linear(embedding,1)
        self.dropout=nn.Dropout(0.3)
        self.embed_size=embedding
    def forward(self,x):
        out=self.wp_wcf(x)
        out_p=out[:,:self.embed_size*2]
        out_c=out[:,self.embed_size*2:]
        out_p_1=self.layer_1_p(out_p)
        out_p_1=self.dropout(out_p_1)
        out_p_1=nn.functional.relu(out_p_1)
        out_p_2=self.layer_2_p(out_p_1)
        out_p_2=self.dropout(out_p_2)
        out_p_2=nn.functional.relu(out_p_2)
        out_p_3=self.layer_3_p(out_p_2)
        out_p_3=self.dropout(out_p_3)
        out_p_3=nn.functional.relu(out_p_3)
        out_p_4=self.layer_4_p(out_p_3)
        # out_c=self.layer_1_c(out_c+out_p)
        # out_c=self.dropout(out_c)
        # out_c=nn.functional.relu(out_c)
        # out_c=self.layer_2_c(out_c+out_p_1)
        # out_c=self.dropout(out_c)
        # out_c=nn.functional.relu(out_c)
        # out_c=self.layer_3_c(out_c+out_p_2)
        # out_c=self.dropout(out_c)
        # out_c=nn.functional.relu(out_c)
        # out_c=self.layer_4_c(out_c+out_p_3)
        return out_p_4
class p_cf_add(nn.Module):
    def __init__(self,embedding,head):
        super().__init__()
        self.wp_wcf=nn.Linear(embedding,4*embedding)
        self.layer_norm=nn.LayerNorm(embedding)
        self.layer_1_p=nn.Linear(embedding*2,embedding*4)
        self.layer_2_p=nn.Linear(embedding*4,embedding*2)
        self.layer_3_p=nn.Linear(embedding*2,embedding)
        self.layer_4_p=nn.Linear(embedding,1)
        self.layer_1_c=nn.Linear(embedding*2,embedding*4)
        self.layer_2_c=nn.Linear(embedding*4,embedding*2)
        self.layer_3_c=nn.Linear(embedding*2,embedding)
        self.layer_4_c=nn.Linear(embedding,1)
        self.dropout=nn.Dropout(0.5)
        self.weight=torch.rand(3,2)
    def forward(self,x):
        m=nn.Softmax(dim=0)(self.weight)    
        out=self.wp_wcf(x)
        out_p=out[:,:self.embed_size*2]
        out_c=out[:,self.embed_size*2:]
        out_p_1=self.layer_1_p(out_p)
        out_p_1=self.dropout(out_p_1)
        out_p_1=nn.functional.relu(out_p_1)
        out_p_2=self.layer_2_p(out_p)
        out_p_2=self.dropout(out_p)
        out_p_2=nn.functional.relu(out_p)
        out_p_3=self.layer_3_p(out_p)
        out_p_3=self.dropout(out_p)
        out_p_3=nn.functional.relu(out_p)
        out_p=self.layer_4_p(out_p)
        out_c=self.layer_1_c(out_c+m[0]*out_p_1)
        out_c=self.dropout(out_c)
        out_c=nn.functional.relu(out_c)
        out_c=self.layer_2_c(out_c+m[1]*out_p_2)
        out_c=self.dropout(out_c)
        out_c=nn.functional.relu(out_c)
        out_c=self.layer_3_c(out_c+m[2]*out_p_3)
        out_c=self.dropout(out_c)
        out_c=nn.functional.relu(out_c)
        out_c=self.layer_4_c(out_c)
        return out_p,out_c


class final(nn.Module):
    def __init__(self,embedding,head,layers):
        super().__init__()
        self.encoder=encoder_layer(embedding,head,layers)
        self.p_cf=p_cf_simple(embedding)
    def forward(self,x):
        out=self.encoder(x)
        out_p=self.p_cf(out)
        return out_p
    

In [54]:
class nueral_net(nn.Module):
    def __init__(self,embedding):
        super().__init__()
        self.seq=nn.Sequential(
            nn.Linear(embedding,embedding*2),
            nn.ReLU(),
            nn.LayerNorm(embedding*2),
            nn.Linear(embedding*2,embedding*4),
            nn.ReLU(),
            nn.LayerNorm(embedding*4),
            nn.Linear(embedding*4,embedding*2),
            nn.ReLU(),
            nn.LayerNorm(embedding*2),
            nn.Linear(embedding*2,embedding),
            nn.ReLU(),
            nn.LayerNorm(embedding),
            nn.Linear(embedding,1)
        )
    def forward(self,x):
        out=self.seq(x)
        return out
mo=final(embeddings_size,num_heads,2).to(d)

In [63]:
criterion= torch.nn.HuberLoss()
optimizer=torch.optim.AdamW(mo.parameters(),lr=0.0001)
num_epochs=10
def loss(preds, labels):
    pred_classes = torch.argmax(preds, dim=1)
    true_classes = torch.argmax(labels, dim=1)
    loss=torch.mean((pred_classes - true_classes) ** 2)
    return loss
def accuracy(preds, labels): 
    preds = (preds >= 0.5).float()
    correct = (preds == labels).sum().item()
    total = labels.size(0)
    return correct / total

In [64]:
#training loop
losses=[]
for epoch in range(10):
    mo.train()
    total_loss = 0
    for batch_embeddings, batch_labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        batch_embeddings, batch_labels = batch_embeddings.to(d), batch_labels.to(d)
        optimizer.zero_grad()
        out_p= mo(batch_embeddings)
        loss_p = criterion(out_p.squeeze(), batch_labels[:, 0])
        total_batch_loss = loss_p
        total_batch_loss.backward()
        optimizer.step()
        total_loss += total_batch_loss.item()
    avg_loss = total_loss / len(train_loader)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f},accuracy: {accuracy(out_p, batch_labels):.4f}")

Epoch 1/10: 100%|██████████| 72/72 [00:10<00:00,  7.03it/s]


Epoch 1, Loss: 0.0819,accuracy: 0.7826


Epoch 2/10: 100%|██████████| 72/72 [00:09<00:00,  7.26it/s]


Epoch 2, Loss: 0.0789,accuracy: 0.9130


Epoch 3/10: 100%|██████████| 72/72 [00:10<00:00,  6.97it/s]


...

Epoch 8, Loss: 0.0765,accuracy: 0.9130


Epoch 9/10: 100%|██████████| 72/72 [00:10<00:00,  6.99it/s]


Epoch 9, Loss: 0.0760,accuracy: 0.7826


Epoch 10/10: 100%|██████████| 72/72 [00:09<00:00,  7.41it/s]

Epoch 10, Loss: 0.0755,accuracy: 0.8696


In [ ]:
import numpy
mo.eval()
actuals=[]
preds=[]
with torch.no_grad():
    for batch_embeddings, batch_labels in tqdm(test_loader, desc=f"Epoch {10}/{10}"):
        batch_embeddings, batch_labels = batch_embeddings.to(d), batch_labels.to(d)
        out_p= mo(batch_embeddings)
        print(f"accuracy: {accuracy(out_p, batch_labels):.4f}")




Epoch 10/10:  31%|███▏      | 5/16 [00:00<00:00, 46.94it/s]

accuracy: 0.6875
accuracy: 0.7969
accuracy: 0.7500
accuracy: 0.7344
accuracy: 0.8594
accuracy: 0.7656
accuracy: 0.7656
accuracy: 0.8281
accuracy: 0.7812


Epoch 10/10: 100%|██████████| 16/16 [00:00<00:00, 48.75it/s]

accuracy: 0.7656
accuracy: 0.7969
accuracy: 0.7656
accuracy: 0.7656
accuracy: 0.7344
accuracy: 0.7812
accuracy: 0.8500
